In [2]:
import pandas as pd
train_df = pd.read_csv(
    'train_data.txt',
    sep=':::',
    engine='python',
    names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION']
)
train_df['GENRE'] = train_df['GENRE'].str.strip()
train_df['DESCRIPTION'] = train_df['DESCRIPTION'].str.strip()
train_df.to_csv('movies.csv', index=False)
print("CSV File Created Successfully!")
print(train_df.head())
import numpy as np
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
def load_movie_data(file_path, has_genre=True):
    """
    Parses the custom ' ::: ' delimited text files into a clean Pandas DataFrame.
    """
    ids, titles, genres, descriptions = [], [], [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(' ::: ')
            if has_genre and len(parts) >= 4:
                ids.append(parts[0])
                titles.append(parts[1])
                genres.append(parts[2])
                descriptions.append(" ::: ".join(parts[3:]))
            elif not has_genre and len(parts) >= 3:
                ids.append(parts[0])
                titles.append(parts[1])
                descriptions.append(" ::: ".join(parts[2:]))
    if has_genre:
        return pd.DataFrame({
            'ID': ids, 'Title': titles, 'Genre': genres, 'Description': descriptions
        })
    else:
        return pd.DataFrame({
            'ID': ids, 'Title': titles, 'Description': descriptions
        })
def clean_text(text):
    """
    Applies standard text cleaning optimizations: lowercasing,
    removing punctuation, digits, and extra whitespaces.
    """
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
print("Loading datasets...")
train_df = load_movie_data('train_data.txt', has_genre=True)
test_df = load_movie_data('test_data_solution.txt', has_genre=True)
print(f"Loaded {len(train_df)} training samples and {len(test_df)} test samples.")
print("Cleaning movie descriptions...")
train_df['Cleaned_Desc'] = train_df['Description'].apply(clean_text)
test_df['Cleaned_Desc'] = test_df['Description'].apply(clean_text)
print("Vectorizing text data using TF-IDF...")
tfidf = TfidfVectorizer(max_features=50000, stop_words='english', sublinear_tf=True)
X_train = tfidf.fit_transform(train_df['Cleaned_Desc'])
y_train = train_df['Genre']
X_test = tfidf.transform(test_df['Cleaned_Desc'])
y_test = test_df['Genre']
print("Training the Logistic Regression classifier...")
model = LogisticRegression(C=1.0, max_iter=1000, solver='saga', n_jobs=-1)
model.fit(X_train, y_train)
print("\nEvaluating model performance...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report (Sample of major genres):")
print(classification_report(y_test, y_pred, zero_division=0))
def predict_genre(title, description):
    cleaned = clean_text(description)
    features = tfidf.transform([cleaned])
    prediction = model.predict(features)[0]
    print(f"\nTitle: {title}")
    print(f"Predicted Genre: {prediction}")
    return prediction
predict_genre(
    "The Cosmic Horizon",
    "A lone astronaut uncovers a dark secret hidden inside an ancient alien artifact orbiting Jupiter, leading to a desperate race against time to save Earth."
)

CSV File Created Successfully!
   ID                               TITLE     GENRE  \
0   1       Oscar et la dame rose (2009)      drama   
1   2                       Cupid (1997)   thriller   
2   3   Young, Wild and Wonderful (1980)      adult   
3   4              The Secret Sin (1915)      drama   
4   5             The Unrecovered (2007)      drama   

                                         DESCRIPTION  
0  Listening in to a conversation between his doc...  
1  A brother and sister with a past incestuous re...  
2  As the bus empties the students for their fiel...  
3  To help their unemployed father make ends meet...  
4  The film's title refers not only to the un-rec...  
Loading datasets...
Loaded 54214 training samples and 30482 test samples.
Cleaning movie descriptions...
Vectorizing text data using TF-IDF...
Training the Logistic Regression classifier...

Evaluating model performance...
Overall Accuracy: 59.26%

Classification Report (Sample of major genres):
           

'sci-fi'